# FloodSense feature preparation and validation splits

Build model inputs and validation splits from the cleaned training data. No held-out Imelda test data is loaded.

## 1. Load the cleaned data and select model inputs

Load the prepared training rows, then select only configured numeric and categorical features plus the target. Coordinates, storm identifiers, and other excluded columns are not model inputs.

In [14]:
import importlib
import sys
from pathlib import Path

project_root = next(
    (parent for parent in (Path.cwd(), *Path.cwd().parents)
     if (parent / "src" / "config.py").is_file()),
    None,
)
if project_root is None:
    raise FileNotFoundError("Could not locate the FloodSense project root from the notebook working directory.")
sys.path.insert(0, str(project_root))

import src.features
src.features = importlib.reload(src.features)

from src.data import load_clean
area_group_splits = src.features.area_group_splits
build_preprocessor = src.features.build_preprocessor
describe_splits = src.features.describe_splits
get_xy = src.features.get_xy
leave_one_storm_out_splits = src.features.leave_one_storm_out_splits

df = load_clean()
X, y = get_xy(df)
print(f"Feature matrix: {X.shape}; target: {y.shape}")
print(f"Model features: {X.columns.tolist()}")


Cleaned training data shape: (66514, 14)
Missing values by column:
year                   0
storm_name             0
storm_date             0
lat                    0
lon                    0
weight                 0
elevation_m            0
slope_degrees          0
distance_to_bayou_m    0
paved_pct              0
land_cover_type        0
rainfall_inches        0
flooded                0
grid_id                0
Feature matrix: (66514, 6); target: (66514,)
Model features: ['elevation_m', 'slope_degrees', 'distance_to_bayou_m', 'paved_pct', 'rainfall_inches', 'land_cover_type']


**My observations:** The cleaned dataset has 66,514 rows and 4,492 grid cells. `X` contains five numeric features (`elevation_m`, `slope_degrees`, `distance_to_bayou_m`, `paved_pct`, and `rainfall_inches`) plus `land_cover_type`; `y` is `flooded`. Coordinates, storm identifiers, `grid_id`. The target classes are nearly balanced overall (about 49% not flooded and 51% flooded).

## 2. Create geographic GroupKFold splits

GroupKFold assigns complete `grid_id` areas to either training or validation. This tests generalization to locations outside the training areas and prevents nearby grid cells from leaking across a split.

In [15]:
area_splits = list(area_group_splits(df, n_splits=5))
describe_splits(area_splits, df)


Split 1: train=53211, validation=13303
  train class counts: {0: 26058, 1: 27153}
  train class proportions: {0: 0.49, 1: 0.51}
  validation class counts: {0: 6501, 1: 6802}
  validation class proportions: {0: 0.489, 1: 0.511}
  grid_id overlap: none
Split 2: train=53211, validation=13303
  train class counts: {0: 26041, 1: 27170}
  train class proportions: {0: 0.489, 1: 0.511}
  validation class counts: {0: 6518, 1: 6785}
  validation class proportions: {0: 0.49, 1: 0.51}
  grid_id overlap: none
Split 3: train=53211, validation=13303
  train class counts: {0: 26013, 1: 27198}
  train class proportions: {0: 0.489, 1: 0.511}
  validation class counts: {0: 6546, 1: 6757}
  validation class proportions: {0: 0.492, 1: 0.508}
  grid_id overlap: none
Split 4: train=53211, validation=13303
  train class counts: {0: 26049, 1: 27162}
  train class proportions: {0: 0.49, 1: 0.51}
  validation class counts: {0: 6510, 1: 6793}
  validation class proportions: {0: 0.489, 1: 0.511}
  grid_id overlap:

**My observations:** The five geographic folds have nearly equal validation sizes (13,302–13,303 rows). Flooded proportions stay close to 51% in both training and validation for every fold. Each split reports no shared `grid_id`, so evaluation tests held-out areas without geographic-group overlap.

### What the two validation strategies measure

The area split holds out new places, so it tests geographic generalization. The storm split holds out a new storm while allowing mostly known places in training, so it tests event generalization. The strict storm variant also removes shared areas; it is included below to show why that setting leaves very little training data.

## 3. Create leave-one-storm-out splits

Hold out each storm in turn to test generalization to a new event. By default, training includes all rows from the other two storms, including rows from places seen during the held-out storm. The overlap report quantifies how many validation areas are also represented in training.

In [16]:
storm_splits = list(leave_one_storm_out_splits(df, drop_overlap_areas=False))
describe_splits(storm_splits, df, allow_overlap=True)


Storm Memorial Day Flood: train=44734, validation=21780
  train class counts: {0: 21777, 1: 22957}
  train class proportions: {0: 0.487, 1: 0.513}
  validation class counts: {0: 10782, 1: 10998}
  validation class proportions: {0: 0.495, 1: 0.505}
  validation grid_ids also in training: 3881/3997 (97.1%)
Storm Tax Day Flood: train=44664, validation=21850
  train class counts: {0: 21833, 1: 22831}
  train class proportions: {0: 0.489, 1: 0.511}
  validation class counts: {0: 10726, 1: 11124}
  validation class proportions: {0: 0.491, 1: 0.509}
  validation grid_ids also in training: 3892/4000 (97.3%)
Storm Hurricane Harvey: train=43630, validation=22884
  train class counts: {0: 21508, 1: 22122}
  train class proportions: {0: 0.493, 1: 0.507}
  validation class counts: {0: 11051, 1: 11833}
  validation class proportions: {0: 0.483, 1: 0.517}
  validation grid_ids also in training: 3907/4022 (97.1%)


**My observations:** Each validation fold contains one complete storm, while training uses all rows from the other two storms. This gives substantially more training data than the strict variant below. Many validation grid cells also appear in training, so this split evaluates a new storm mostly in known places rather than new geographic areas. Validation classes are close to balanced.

### Strict storm split comparison

For evidence, create the strict version that removes training rows whose `grid_id` occurs in the validation storm. The overlap assertion should pass, while the resulting training sets are expected to be much smaller.

In [17]:
strict_storm_splits = list(leave_one_storm_out_splits(df, drop_overlap_areas=True))
describe_splits(strict_storm_splits, df)


Storm Memorial Day Flood: train=1592, validation=21780
  train class counts: {0: 1504, 1: 88}
  train class proportions: {0: 0.945, 1: 0.055}
  validation class counts: {0: 10782, 1: 10998}
  validation class proportions: {0: 0.495, 1: 0.505}
  grid_id overlap: none
Storm Tax Day Flood: train=1596, validation=21850
  train class counts: {0: 1562, 1: 34}
  train class proportions: {0: 0.979, 1: 0.021}
  validation class counts: {0: 10726, 1: 11124}
  validation class proportions: {0: 0.491, 1: 0.509}
  grid_id overlap: none
Storm Hurricane Harvey: train=1542, validation=22884
  train class counts: {0: 1504, 1: 38}
  train class proportions: {0: 0.975, 1: 0.025}
  validation class counts: {0: 11051, 1: 11833}
  validation class proportions: {0: 0.483, 1: 0.517}
  grid_id overlap: none


## 4. Fit preprocessing inside one fold

The preprocessor returned by `build_preprocessor()` is unfitted. Fit a new instance using only the training rows for each fold, then transform its validation rows; never fit preprocessing on validation data.

In [18]:
train_idx, val_idx = area_splits[0]
fold_preprocessor = build_preprocessor()
X_train = fold_preprocessor.fit_transform(X.iloc[train_idx])
X_validation = fold_preprocessor.transform(X.iloc[val_idx])
print(f"Transformed training shape: {X_train.shape}")
print(f"Transformed validation shape: {X_validation.shape}")


Transformed training shape: (53211, 9)
Transformed validation shape: (13303, 9)


**My observations:** The preprocessor was fitted on the first geographic fold’s training rows, then used to transform that fold’s validation rows. Both transformed arrays have nine columns: five standardized numeric features and four one-hot land-cover categories. Because fitting happened before transforming validation data, validation values did not contribute to the learned scaling statistics or category encoding.